# Supplementary Tables
## Gene and transcript counts across species
### Author: Martin Loza
### Date: 25/12/12

Let's create a table to summarize the number of selected genes across species

In [33]:
# Change R language to English
Sys.setenv(LANGUAGE = "en")

# Init
suppressPackageStartupMessages({
    library(dplyr)
    library(stringr)
})

# Local variables
seed = 777
date = "251212"

in_dir = "/Users/martin/Documents/Projects/lncRNA_TF_pairs_analysis/Data/ENSEMBL/selected/"
out_dir = "/Users/martin/Documents/Projects/lncRNA_TF_pairs_analysis/Data/Supplementary/"

### Load the data

In [34]:
# Load the transcripts data accross species

# Create a list to store the species data
data_list = list()  
# Search for the file related to the species
files <- list.files(in_dir)

# Loop through each file
for (file_name in files) {
    # Extract species name from the file name
    species_name <- str_split(file_name, pattern = "_")[[1]][1]
    cat("Processing species:", species_name, "\n")
    # Load the transcripts data
    transcripts = read.table(paste0(in_dir,file_name), sep = "\t", header = TRUE, 
                             stringsAsFactors = FALSE, quote = "", comment.char = "", 
                             fill = TRUE )
    # Store the data in the list
    data_list[[species_name]] = transcripts
}

Processing species: armadillo 
Processing species: chicken 
Processing species: dog 
Processing species: drosophila 
Processing species: elegans 
Processing species: ferret 
Processing species: human 
Processing species: macaque 
Processing species: mouse 
Processing species: rat 
Processing species: zebrafish 


In [35]:
names(data_list)
head(data_list[["human"]])

[1] "armadillo"  "chicken"    "dog"        "drosophila" "elegans"   
 [6] "ferret"     "human"      "macaque"    "mouse"      "rat"       
[11] "zebrafish"

,Chromosome.scaffold.name,Gene.start..bp.,Gene.end..bp.,Strand,Gene.stable.ID,Transcript.stable.ID,Transcript.start..bp.,Transcript.end..bp.,Transcript.type,Gene.type,Gene.name,Gene.description,TSS,is_pcg,is_ncrna
,<chr>,<int>,<int>,<int>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<int>,<lgl>,<lgl>
1,1,11121,24894,1,ENSG00000290825,ENST00000456328,11850,14416,lncRNA,lncRNA,DDX11L16,DEAD/H-box helicase 11 like 16 (pseudogene) [Source:NCBI gene (formerly Entrezgene);Acc:727856],11850,FALSE,TRUE
2,1,11121,24894,1,ENSG00000290825,ENST00000832823,14404,24894,lncRNA,lncRNA,DDX11L16,DEAD/H-box helicase 11 like 16 (pseudogene) [Source:NCBI gene (formerly Entrezgene);Acc:727856],14404,FALSE,TRUE
3,1,11121,24894,1,ENSG00000290825,ENST00000832824,11121,14413,lncRNA,lncRNA,DDX11L16,DEAD/H-box helicase 11 like 16 (pseudogene) [Source:NCBI gene (formerly Entrezgene);Acc:727856],11121,FALSE,TRUE
4,1,11121,24894,1,ENSG00000290825,ENST00000832825,11125,14405,lncRNA,lncRNA,DDX11L16,DEAD/H-box helicase 11 like 16 (pseudogene) [Source:NCBI gene (formerly Entrezgene);Acc:727856],11125,FALSE,TRUE
5,1,11121,24894,1,ENSG00000290825,ENST00000832826,11410,14413,lncRNA,lncRNA,DDX11L16,DEAD/H-box helicase 11 like 16 (pseudogene) [Source:NCBI gene (formerly Entrezgene);Acc:727856],11410,FALSE,TRUE
6,1,11121,24894,1,ENSG00000290825,ENST00000832827,11411,14413,lncRNA,lncRNA,DDX11L16,DEAD/H-box helicase 11 like 16 (pseudogene) [Source:NCBI gene (formerly Entrezgene);Acc:727856],11411,FALSE,TRUE


Select only PCG or ncRNA

In [36]:
# Setup the data
data_list <- lapply(data_list, function(x){
    # Select only PCGs and ncRNAs
    x <- x %>% filter(is_pcg == TRUE | is_ncrna == TRUE)
    # Set TSS as integer
    x$TSS <- as.integer(x$TSS)
    # Set Strand as character
    x$Strand <- as.character(x$Strand)
    return(x)
})

Let's define the order of the genes

In [37]:
sel_genes <- c('protein_coding',
  'ncRNA', 
  'lncRNA', 
  'miRNA', 'lincRNA', 'pre_miRNA',
  'scRNA', 'snRNA', 'snoRNA', 'scaRNA', 'sRNA', "piRNA",
  'misc_RNA', 
  'Mt_tRNA', 'Mt_rRNA', 'rRNA', 'rRNA_pseudogene', "tRNA",
  'processed_transcript', 'ribozyme'
)

### Summary Table 1: Gene-level counts (unique genes by Gene.stable.ID)

In [38]:
# Create gene-level summary table
gene_summary_list = list()

for (species_name in names(data_list)) {
    cat("Processing gene counts for:", species_name, "\n")
    
    # Get unique genes only
    unique_genes = data_list[[species_name]] %>%
        distinct(Gene.stable.ID, .keep_all = TRUE)

    # Filter to selected gene types
    unique_genes = unique_genes %>%
        filter(Gene.type %in% sel_genes)
    
    # Count by gene type
    gene_counts = unique_genes %>%
        group_by(Gene.type) %>%
        summarise(count = n()) %>%
        arrange(desc(count))
    
    # Add species column
    gene_counts$species = species_name
    
    # Store in list
    gene_summary_list[[species_name]] = gene_counts
}

# Combine all species into one table
gene_summary_table = bind_rows(gene_summary_list)

# Reshape to wide format for better visualization
gene_summary_wide = gene_summary_table %>%
    tidyr::pivot_wider(names_from = species, values_from = count, values_fill = 0) %>%
    dplyr::rename(Gene_biotype = Gene.type) %>%
    arrange(match(Gene_biotype, order_genes))

print(gene_summary_wide)

Processing gene counts for: armadillo 
Processing gene counts for: chicken 
Processing gene counts for: dog 
Processing gene counts for: drosophila 
Processing gene counts for: elegans 
Processing gene counts for: ferret 
Processing gene counts for: human 
Processing gene counts for: macaque 
Processing gene counts for: mouse 
Processing gene counts for: rat 
Processing gene counts for: zebrafish 
# A tibble: 19 × 12
   Gene_biotype  armadillo chicken   dog drosophila elegans ferret human macaque
   <chr>             <int>   <int> <int>      <int>   <int>  <int> <int>   <int>
 1 protein_codi…     22711   17007 20500      13986   19985  19910 22847   21591
 2 ncRNA                 0       0     0       3304    7764      0     0       0
 3 lncRNA                0   11944  6479          0       0      0 36443    4769
 4 miRNA               841     674   986          0     261    818  1945     626
 5 lincRNA            3179       0     0          0     194   8246     0       0
 6 scRNA    

### Summary Table 2: Transcript-level counts (by Transcript.stable.ID)

In [39]:
# Create transcript-level summary table
transcript_summary_list = list()

for (species_name in names(data_list)) {
    cat("Processing transcript counts for:", species_name, "\n")
    
    # Count by transcript type (all transcripts, no deduplication needed)
    transcript_counts = data_list[[species_name]] %>%
        group_by(Transcript.type) %>%
        summarise(count = n()) %>%
        arrange(desc(count))
    
    # Add species column
    transcript_counts$species = species_name
    
    # Store in list
    transcript_summary_list[[species_name]] = transcript_counts
}

# Combine all species into one table
transcript_summary_table = bind_rows(transcript_summary_list)

# Reshape to wide format for better visualization
transcript_summary_wide = transcript_summary_table %>%
    tidyr::pivot_wider(names_from = species, values_from = count, values_fill = 0) %>%
    dplyr::rename(Transcript_biotype = Transcript.type) %>%
    arrange(match(Transcript_biotype, order_genes))

print(transcript_summary_wide)

Processing transcript counts for: armadillo 
Processing transcript counts for: chicken 
Processing transcript counts for: dog 
Processing transcript counts for: drosophila 
Processing transcript counts for: elegans 
Processing transcript counts for: ferret 
Processing transcript counts for: human 
Processing transcript counts for: macaque 
Processing transcript counts for: mouse 
Processing transcript counts for: rat 
Processing transcript counts for: zebrafish 
# A tibble: 20 × 12
   Transcript_biotype   armadillo chicken   dog drosophila elegans ferret  human
   <chr>                    <int>   <int> <int>      <int>   <int>  <int>  <int>
 1 protein_coding           26551   44876 43458      30802   31865  20062 221639
 2 ncRNA                        0       0     0       3060    8440      0      0
 3 lncRNA                       0   26656  7735          0       0      0 195159
 4 miRNA                      841     674   986        485     458    818   1945
 5 lincRNA                 

In [40]:
# Save the summary tables
write.table(gene_summary_wide, file = paste0(out_dir, "Supplementary_Table_gene_counts_", date, ".tsv"), 
            sep = "\t", quote = FALSE, row.names = FALSE)
write.table(transcript_summary_wide, file = paste0(out_dir, "Supplementary_Table_transcript_counts_", date, ".tsv"), 
            sep = "\t", quote = FALSE, row.names = FALSE)